# 3 – Autocorrelazione spaziale e LISA (Tanzania)

**SPEC 2026** – Econometria Spaziale (Python Lab)

Equivalente Python di `R/3_lisa.R`

- Pesi spaziali: contiguità Queen
- Moran’s I globale (analitico + Monte Carlo)
- LISA (Local Moran’s I) e mappa cluster
- Moran scatterplot

In [ ]:
!pip install -q geopandas pyarrow libpysal esda splot mapclassify

In [ ]:
import geopandas as gpd
import numpy as np
import os
import matplotlib.pyplot as plt
from shapely.geometry import LineString
from libpysal.weights import Queen, lag_spatial
from esda import Moran, Moran_Local
import warnings
warnings.filterwarnings('ignore')
np.random.seed(123)

BASE = "https://github.com/vincnardelli/spec/raw/main/data"
for f in ["tanzania.parquet", "kc_house.parquet", "kc_grid.parquet",
          "italian_provinces.parquet", "visium_hne_points.parquet"]:
    if not os.path.exists(f):
        !wget -q {BASE}/{f}

## 1) Caricamento dati

In [ ]:
tz = gpd.read_parquet("tanzania.parquet")
tz.head()

## 2) Visualizzazione esplorativa

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

tz.plot(column="FEFRTRWTFR", legend=True, cmap="YlOrRd",
        legend_kwds={"label": "Fertility rate"}, ax=axes[0])
axes[0].set_title("Coropleta")
axes[0].set_axis_off()

axes[1].hist(tz["FEFRTRWTFR"], bins=5, color="#d7301f", edgecolor="white")
axes[1].set_xlabel("Fertility rate")
axes[1].set_ylabel("Count")
axes[1].set_title("Distribuzione")

plt.tight_layout()
plt.show()

## 3) Pesi spaziali – Queen contiguity

In [ ]:
w = Queen.from_dataframe(tz, silence_warnings=True)
w.transform = "r"
print(w)

In [ ]:
centroids = tz.geometry.centroid
lines = []
for i in w.neighbors:
    for j in w.neighbors[i]:
        lines.append(LineString([centroids.iloc[i], centroids.iloc[j]]))
network = gpd.GeoDataFrame(geometry=lines, crs=tz.crs)

fig, ax = plt.subplots(figsize=(8, 8))
tz.plot(ax=ax, facecolor="#f0f0f0", edgecolor="grey")
network.plot(ax=ax, color="#404040", linewidth=0.5, alpha=0.8)
ax.set_title("Rete di contiguit\u00e0 Queen")
ax.set_axis_off()
plt.tight_layout()
plt.show()

## 4) Moran’s I globale

In [ ]:
y = tz["FEFRTRWTFR"].values

mi = Moran(y, w, permutations=999)
print(f"Moran's I:       {mi.I:.4f}")
print(f"Expected I:      {mi.EI:.4f}")
print(f"p-value (norm):  {mi.p_norm:.4f}")
print(f"p-value (sim):   {mi.p_sim:.4f}")
print(f"z-score (norm):  {mi.z_norm:.4f}")

## 5) LISA (Local Moran’s I)

In [ ]:
lisa = Moran_Local(y, w, permutations=999)

q_labels = {1: "High-High", 2: "Low-High", 3: "Low-Low", 4: "High-Low"}
tz["lisa_cluster"] = [
    q_labels[q] if p < 0.05 else "Not significant"
    for q, p in zip(lisa.q, lisa.p_sim)
]

z_y = (y - y.mean()) / y.std()
tz["z_y"] = z_y
tz["lag_z_y"] = lag_spatial(w, z_y)

In [ ]:
colors = {
    "High-High": "#B2182B", "Low-Low": "#2166AC",
    "High-Low": "#EF8A62",  "Low-High": "#67A9CF",
    "Not significant": "#d9d9d9"
}

fig, ax = plt.subplots(figsize=(8, 8))
tz.plot(color=tz["lisa_cluster"].map(colors), edgecolor="white",
        linewidth=0.3, ax=ax)
handles = [plt.Rectangle((0,0),1,1, facecolor=c) for c in colors.values()]
ax.legend(handles, colors.keys(), title="LISA cluster", loc="lower left")
ax.set_title("LISA Cluster Map \u2013 Fertility Rate")
ax.set_axis_off()
plt.tight_layout()
plt.show()

## 6) Moran scatterplot

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.axhline(0, color="grey", lw=0.5)
ax.axvline(0, color="grey", lw=0.5)

for label, color in colors.items():
    mask = tz["lisa_cluster"] == label
    ax.scatter(tz.loc[mask, "z_y"], tz.loc[mask, "lag_z_y"],
               c=color, label=label, s=40, alpha=0.9, edgecolors="k", lw=0.3)

xr = np.array([-3, 3])
ax.plot(xr, mi.I * xr, color="grey", lw=1)

ax.set_xlabel("z(Fertility rate)")
ax.set_ylabel("Wz(Fertility rate)")
ax.set_title("Moran scatterplot")
ax.legend(title="LISA cluster", fontsize=8)
plt.tight_layout()
plt.show()